In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("source_table", "load_report_log_cpc_Stg_Elig", "Source Table")
dbutils.widgets.text("destination_table", "Load_Report_Log_CMC_CPC_Attr", "Destination Table")
dbutils.widgets.text("error_source_table", "error_load_report_log_cpc_Stg_Elig", "Error Source Table")
dbutils.widgets.text("error_destination_table", "load_error_report_cmc_cpc_attr", "Error Destination Table")
dbutils.widgets.text('catalog', 'oh_apm_stg')  
dbutils.widgets.text('schema_name', 'vendor_extracts')  
dbutils.widgets.text('schema_name_ApmRprt', 'apm_report')  

In [0]:
source_table = dbutils.widgets.get("source_table")
destination_table = dbutils.widgets.get("destination_table")
catalog = dbutils.widgets.get('catalog')
schema_name = dbutils.widgets.get('schema_name')
schema_name_ApmRprt = dbutils.widgets.get('schema_name_ApmRprt')
#
print(f"📌 Catalog: {catalog}")
print(f"📌 Schema Name: {schema_name}")
print(f"📌 APM Report Schema Name: {schema_name_ApmRprt}")
print()
print(f"📌 Source Table: {source_table}")
print(f"📌 Destination Table: {destination_table}")
error_source_table = dbutils.widgets.get("error_source_table")
error_destination_table = dbutils.widgets.get("error_destination_table")
print(f"📌 Error Source Table: {error_source_table}")
print(f"📌 Error Destination Table: {error_destination_table}")

In [0]:
try:
    # Get the flag from Prevalidation
    run_flag = dbutils.jobs.taskValues.get(taskKey="Pre_validation", key="run_flag", debugValue=False)
    print(f"✅ Retrieved run_flag: {run_flag}")

    if run_flag:
        print("🚀 Flag is True. Proceeding with data copy...")

        # Get dates from DateProvider
        start_date = dbutils.jobs.taskValues.get(taskKey="End_Date_time", key="start_date", debugValue="1900-01-01")
        end_date = dbutils.jobs.taskValues.get(taskKey="End_Date_time", key="end_date", debugValue="1900-01-01")

        print(f"📌 Start Date: {start_date}")
        print(f"📌 End Date: {end_date}")

        # Run the SQL with WHERE clause to filter by dates
        spark.sql(f"""
            INSERT INTO {catalog}.{schema_name_ApmRprt}.{destination_table}
            SELECT
                File_Name,
                Date_Received_by_APM,
                Status AS Load_Status,
                Number_of_Records_Received,
                Number_of_Records_Loaded,
                Number_of_Error_Records,
                TO_DATE(Start_Load_Date, 'yyyy-MM-dd HH:mm:ss') AS Start_Load_Date,
                TO_DATE(End_Load_Date, 'yyyy-MM-dd HH:mm:ss') AS End_Load_Date
            FROM {catalog}.{schema_name}.{source_table}
            WHERE Start_Load_Date = '{start_date}' AND End_Load_Date = '{end_date}'
        """)

        print("✅ Data successfully copied to destination table.")
    else:
        print("⚠️ Flag is False. Skipping data copy.")

except Exception as e:
    print("❌ An error occurred during the CopyToAPMTables task.")
    print(f"Error details: {str(e)}")
    raise


In [0]:
try:
    # Get the error_load_report_flag from Prevalidation
    error_load_report_flag = dbutils.jobs.taskValues.get(taskKey="Pre_validation", key="error_load_report_flag", debugValue=False)
    print(f"✅ Retrieved error_load_report_flag: {error_load_report_flag}")

    if error_load_report_flag:
        print("🚀 error_load_report_flag is True. Proceeding with Error Load Report data copy...")

        # Run simple insert into destination table
        spark.sql(f"""
            INSERT INTO {catalog}.{schema_name_ApmRprt}.{error_destination_table}
            SELECT *
            FROM {catalog}.{schema_name}.{error_source_table}
        """)
        print("✅ Error Load Report data successfully copied to destination table.")
    else:
        print("⚠️ error_load_report_flag is False. Skipping Error Load Report data copy.")

except Exception as e:
    print("❌ An error occurred during the Error Load Report Copy task.")
    print(f"Error details: {str(e)}")
    raise

In [0]:
%sql
select *
from ${catalog}.${schema_name}.${source_table}
;

In [0]:
%sql
select *
from ${catalog}.${schema_name}.${error_source_table}
;

In [0]:
%sql
select *
  from ${catalog}.${schema_name_ApmRprt}.${destination_table}
where file_name RLIKE 'ODM\.EDW\.VEN101FA|ODM\.EDW\.VEN116FA'
order by Start_Load_Date desc, File_Name
limit 200
;

In [0]:
%sql
select *
  from ${catalog}.${schema_name_ApmRprt}.${error_destination_table}
where file_name RLIKE 'ODM\.EDW\.VEN101FA|ODM\.EDW\.VEN116FA'
order by Start_Load_Date desc, File_Name
limit 200